In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação térmica (RandomForestRegressor) aprendida apenas com dados sem falha
+ Correções físicas (offset, ganho, tilt, shift)
+ Classificação de falhas (RandomForestClassifier)
+ Split por temperatura SEM sobreposição (generalização térmica real)
Autor: Luiz Eduardo Abdala José
"""

import re, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)

# ========= PARÂMETROS =========
ARQ_BASE = "base-completo--.pkl"   # ✅ base com 3 classes
REF_TEMP = 20
FREQ_MIN_KHZ = 30
FREQ_MAX_KHZ = 120
SMOOTH_WIN = 5
TAU_MAX_FRAC = 0.025
ANCHOR_TO_REF_ENDS = True
CAPS = dict(gain_frac=0.60, offset_frac=0.60, tilt_frac=0.40)

RF_CLASSIF_PARAMS = dict(
    n_estimators=300,
    max_depth=9,
    min_samples_split=4,
    min_samples_leaf=3,
    random_state=0,
    n_jobs=-1
)

# ========= FUNÇÕES AUXILIARES =========
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None and fmin_khz <= f/1e3 <= fmax_khz:
            cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs)[order]

def slope_over_band(f, x):
    return float((x[-1] - x[0]) / (f[-1] - f[0] + 1e-12))

def energy_weighted_centroid(f, x):
    xm = np.asarray(x, float)
    w = xm * xm
    den = float(np.trapezoid(w, f))
    if den <= 1e-18: return float(np.mean(f))
    num = float(np.trapezoid(f * w, f))
    return num / den

def compute_features(X, f):
    X = np.asarray(X, float)
    n, m = X.shape
    out = []
    for i in range(n):
        x = X[i]
        mean = float(np.mean(x))
        std = float(np.std(x))
        amp = float(x.max() - x.min())
        slope = slope_over_band(f, x)
        centroid = energy_weighted_centroid(f, x)
        out.append([mean, std, amp, slope, centroid])
    cols = ["mean", "std", "amp", "slope", "centroid"]
    return np.array(out, float), cols

def fit_feature_vs_temp_models(F, T):
    models = {}
    from sklearn.ensemble import RandomForestRegressor
    T = np.asarray(T).reshape(-1,1)
    names = ["mean", "std", "amp", "slope", "centroid"]
    for j, name in enumerate(names):
        rf = RandomForestRegressor(
            n_estimators=300, max_depth=8,
            min_samples_split=4, min_samples_leaf=3,
            random_state=42, n_jobs=-1
        )
        rf.fit(T, F[:, j])
        models[name] = rf
    return models

def feature_targets_at_ref(models, ref_temp=REF_TEMP):
    Tref = np.array([[ref_temp]])
    return {name: float(m.predict(Tref)[0]) for name, m in models.items()}

def apply_compensation_by_features(x, f, targets, caps, y_ref=None):
    x = x.copy()
    mean_t = targets["mean"]; amp_t = targets["amp"]
    slope_t = targets["slope"]; centroid_t = targets["centroid"]
    mean_x = float(x.mean()); amp_x = float(x.max() - x.min())
    slope_x = slope_over_band(f, x)

    offset = np.clip(mean_t - mean_x, -caps["offset_frac"]*amp_x, caps["offset_frac"]*amp_x)
    x += offset
    gain = amp_t / max(amp_x, 1e-9)
    gain = np.clip(gain, 1.0 - caps["gain_frac"], 1.0 + caps["gain_frac"])
    x = mean_t + gain*(x - mean_t)
    delta_slope = slope_t - slope_x
    tilt_signal = np.linspace(-0.5,0.5,len(x))*(delta_slope*(f[-1]-f[0]))
    tilt_signal = np.clip(tilt_signal, -caps["tilt_frac"]*amp_x, caps["tilt_frac"]*amp_x)
    x += tilt_signal
    cent_x = energy_weighted_centroid(f, x)
    delta_c = centroid_t - cent_x
    tau = np.clip(delta_c, -TAU_MAX_FRAC*(f[-1]-f[0]), TAU_MAX_FRAC*(f[-1]-f[0]))
    if abs(tau) > 1e-12:
        f_shift = f + tau
        x = np.interp(f, f_shift, x, left=x[0], right=x[-1])
    if ANCHOR_TO_REF_ENDS and (y_ref is not None):
        e0 = x[0] - y_ref[0]; e1 = x[-1] - y_ref[-1]
        corr = np.linspace(e0, e1, len(x))
        x -= corr
    return x

# ========= ETAPA 1 – CARREGAMENTO =========
df = pd.read_pickle(ARQ_BASE)
fcols, fhz = get_freq_columns(df, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
fhz_khz = fhz/1e3
df_sem = df[df["falha"] == 0].copy()

# ========= ETAPA 2 – REFERÊNCIA @20°C =========
pool_ref = df_sem.loc[np.isclose(df_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)
y_ref = np.median(pool_ref, axis=0) if len(pool_ref)>0 else np.median(df_sem[fcols], axis=0)

# ========= ETAPA 3 – TREINO COMPENSAÇÃO =========
X_sem = df_sem[fcols].to_numpy(float)
T_sem = df_sem["temperatura_c"].to_numpy(float)
F_sem,_ = compute_features(X_sem, fhz)
feat_models = fit_feature_vs_temp_models(F_sem, T_sem)
targets = feature_targets_at_ref(feat_models, REF_TEMP)

# ========= ETAPA 4 – COMPENSAÇÃO =========
Y_comp = np.zeros_like(df[fcols].to_numpy(float))
for i in range(len(df)):
    Y_comp[i] = apply_compensation_by_features(df.iloc[i][fcols].to_numpy(float), fhz, targets, CAPS, y_ref)

df_comp = df.copy(); df_comp[fcols] = Y_comp

# ========= ETAPA 5 – SPLIT SEM REPETIÇÃO DE TEMPERATURAS =========
temps_all = sorted(df_comp["temperatura_c"].unique())
# Exemplo: alterna metade das temperaturas para treino e metade para teste
temps_train = temps_all[::2]
temps_test  = temps_all[1::2]

print(f"Temperaturas treino: {temps_train}")
print(f"Temperaturas teste:  {temps_test}")

df_train = df_comp[df_comp["temperatura_c"].isin(temps_train)]
df_test  = df_comp[df_comp["temperatura_c"].isin(temps_test)]

X_train = df_train[fcols].to_numpy(float)
y_train = df_train["falha"].to_numpy(int)
X_test  = df_test[fcols].to_numpy(float)
y_test  = df_test["falha"].to_numpy(int)

# ========= ETAPA 6 – CLASSIFICAÇÃO =========
clf = RandomForestClassifier(**RF_CLASSIF_PARAMS)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n== RESULTADOS RANDOM FOREST (RF + FEATURES, split sem overlap) ==")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))


In [ ]:
# ========= ETAPA 7 – GRÁFICO DE EXEMPLO =========
print("\n🔹 Gerando gráfico de exemplo...")
idx_show = df_test.index[10] if len(df_test) > 10 else df_test.index[0]

plt.figure(figsize=(9,5))
plt.plot(fhz_khz, y_ref, '--', c='black', lw=1.2, label=f"Referência {REF_TEMP}°C (sem falha)")
plt.plot(fhz_khz, df.loc[idx_show, fcols], c='tab:red', alpha=0.6,
         label=f"Original {df.loc[idx_show,'temperatura_c']}°C (falha={df.loc[idx_show,'falha']})")
plt.plot(fhz_khz, df_comp.loc[idx_show, fcols], c='tab:blue', lw=2,
         label=f"Compensado {df.loc[idx_show,'temperatura_c']}°C")
plt.title(f"Compensação RF — {FREQ_MIN_KHZ}-{FREQ_MAX_KHZ} kHz")
plt.xlabel("Frequência (kHz)")
plt.ylabel("Parte real da impedância")
plt.legend()
plt.tight_layout()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.spatial.distance import cosine
from math import acos, degrees

def calc_metrics(y_true, y_pred):
    """Calcula todas as métricas entre curvas"""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    corr = np.corrcoef(y_true, y_pred)[0,1]
    # SAM (Spectral Angle Mapper)
    sam_rad = acos(np.clip(np.dot(y_true, y_pred) /
                           (np.linalg.norm(y_true) * np.linalg.norm(y_pred) + 1e-12), -1, 1))
    sam_deg = degrees(sam_rad)
    nrmse = rmse / (y_true.max() - y_true.min() + 1e-12)
    rmsd = np.sqrt(np.mean((y_true - y_pred - np.mean(y_true - y_pred))**2))
    ccdm = 1 - corr
    return dict(R2=r2, RMSE=rmse, MAE=mae, Corr=corr,
                SAM_deg=sam_deg, NRMSE=nrmse, RMSD=rmsd, CCDM=ccdm)

# ===== Exemplo de uso =====
# y_ref: referência 20°C (ex: y_ref do código principal)
# X_orig: curva original (ex: df.loc[idx_show, fcols])
# X_comp: curva compensada (ex: df_comp.loc[idx_show, fcols])

y_ref_vec = y_ref
X_orig = df.loc[idx_show, fcols].to_numpy(float)
X_comp = df_comp.loc[idx_show, fcols].to_numpy(float)

print("\n== MÉTRICAS ORIGINAL vs REFERÊNCIA ==")
print(calc_metrics(y_ref_vec, X_orig))

print("\n== MÉTRICAS COMPENSADO vs REFERÊNCIA ==")
print(calc_metrics(y_ref_vec, X_comp))
